In [1]:
!pip install numpy==1.26.4


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 102.6 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.


In [1]:
pip install git+https://github.com/mimoralea/gym-walk#egg=gym-walk

  Cloning https://github.com/mimoralea/gym-walk to /tmp/pip-install-ld3pvxta/gym-walk_ef789c42171849f6907868b8198afe3f
  Running command git clone --filter=blob:none --quiet https://github.com/mimoralea/gym-walk /tmp/pip-install-ld3pvxta/gym-walk_ef789c42171849f6907868b8198afe3f
  Resolved https://github.com/mimoralea/gym-walk to commit b915b94cf2ad16f8833a1ad92ea94e88159279f5
  Preparing metadata (setup.py) ... done
  Created wheel for gym-walk: filename=gym_walk-0.0.2-py3-none-any.whl size=5377 sha256=c8913db950307db6bbbf612577f868a5ad7dbacad5d356cb4f59e234ec1b89cb
  Stored in directory: /tmp/pip-ephem-wheel-cache-oql4orlv/wheels/bf/23/e5/a94be4a90dd18f7ce958c21f192276cb01ef0daaf2bc66583b
Successfully built gym-walk


In [4]:
import warnings ; warnings.filterwarnings('ignore')

import gym, gym_walk
import numpy as np

import random
import warnings

warnings.filterwarnings('ignore', category=DeprecationWarning)
np.set_printoptions(suppress=True)
random.seed(123); np.random.seed(123)

In [5]:
def print_policy(pi, P, action_symbols=('<', 'v', '>', '^'), n_cols=4, title='Policy:'):
    print(title)
    arrs = {k:v for k,v in enumerate(action_symbols)}
    for s in range(len(P)):
        a = pi[s]
        print("| ", end="")
        if np.all([done for action in P[s].values() for _, _, _, done in action]):
            print("".rjust(9), end=" ")
        else:
            print(str(s).zfill(2), arrs[a].rjust(6), end=" ")
        if (s + 1) % n_cols == 0: print("|")

In [6]:
def print_state_value_function(V, P, n_cols=4, prec=3, title='State-value function:'):
    print(title)
    for s in range(len(P)):
        v = V[s]
        print("| ", end="")
        if np.all([done for action in P[s].values() for _, _, _, done in action]):
            print("".rjust(9), end=" ")
        else:
            print(str(s).zfill(2), '{}'.format(np.round(v, prec)).rjust(6), end=" ")
        if (s + 1) % n_cols == 0: print("|")

In [30]:
def probability_success(env, pi, goal_state, n_episodes=100, max_steps=200):
    random.seed(123); np.random.seed(123) ; env.seed(123)
    results = []
    for _ in range(n_episodes):
        state, done, steps = env.reset(), False, 0
        while not done and steps < max_steps:
            action = pi[state]
            state, _, done, _ = env.step(action)
            steps += 1
        results.append(state == goal_state)
    return np.sum(results)/len(results)

In [40]:
def mean_return(env, pi, n_episodes=100, max_steps=200):
    random.seed(123); np.random.seed(123) ; env.seed(123)
    results = []
    for _ in range(n_episodes):
        state, done, steps = env.reset(), False, 0
        results.append(0.0)
        while not done and steps < max_steps:
            action = pi[state]
            state, reward, done, _ = env.step(action)
            results[-1] += reward
            steps += 1
    return np.mean(results)

In [41]:
env = gym.make('FrozenLake-v1')
P = env.env.P
init_state = env.reset()
goal_state = 15
#LEFT, RIGHT = range(2)

In [42]:
P

{0: {0: [(0.3333333333333333, 0, 0.0, False),
   (0.3333333333333333, 0, 0.0, False),
   (0.3333333333333333, 4, 0.0, False)],
  1: [(0.3333333333333333, 0, 0.0, False),
   (0.3333333333333333, 4, 0.0, False),
   (0.3333333333333333, 1, 0.0, False)],
  2: [(0.3333333333333333, 4, 0.0, False),
   (0.3333333333333333, 1, 0.0, False),
   (0.3333333333333333, 0, 0.0, False)],
  3: [(0.3333333333333333, 1, 0.0, False),
   (0.3333333333333333, 0, 0.0, False),
   (0.3333333333333333, 0, 0.0, False)]},
 1: {0: [(0.3333333333333333, 1, 0.0, False),
   (0.3333333333333333, 0, 0.0, False),
   (0.3333333333333333, 5, 0.0, True)],
  1: [(0.3333333333333333, 0, 0.0, False),
   (0.3333333333333333, 5, 0.0, True),
   (0.3333333333333333, 2, 0.0, False)],
  2: [(0.3333333333333333, 5, 0.0, True),
   (0.3333333333333333, 2, 0.0, False),
   (0.3333333333333333, 1, 0.0, False)],
  3: [(0.3333333333333333, 2, 0.0, False),
   (0.3333333333333333, 1, 0.0, False),
   (0.3333333333333333, 0, 0.0, False)]},
 2:

In [43]:
def decay_schedule(
    init_value, min_value, decay_ratio,
    max_steps, log_start=-2, log_base=10):

    # number of points for the decay curve
    decay_steps = int(max_steps * decay_ratio)

    # logarithmically spaced values between log_start and 0
    log_space = np.logspace(log_start, 0, decay_steps, base=log_base)

    # normalize to start at 1 and go down smoothly
    log_space = (log_space - log_space.min()) / (log_space.max() - log_space.min())

    # reverse so it starts at 1 → 0
    log_space = 1 - log_space

    # scale between init_value and min_value
    values = min_value + (init_value - min_value) * log_space

    # pad with min_value until max_steps
    if decay_steps < max_steps:
        values = np.concatenate([values, np.full(max_steps - decay_steps, min_value)])

    return values


In [35]:
schedule = decay_schedule(init_value=1.0, min_value=0.1, decay_ratio=0.5, max_steps=20)
print(np.round(schedule, 3))

[1.    0.994 0.984 0.967 0.939 0.892 0.813 0.682 0.464 0.1   0.1   0.1
 0.1   0.1   0.1   0.1   0.1   0.1   0.1   0.1  ]


In [44]:
from itertools import count

def generate_trajectory(
    select_action, Q, epsilon,
    env, max_steps=200):

    state, done = env.reset(), False
    trajectory = []

    for t in count():  # infinite counter, but we’ll break
        if t >= max_steps or done:
            break

        # choose action using the given policy (epsilon-greedy usually)
        action = select_action(state, Q, epsilon)

        # take step
        next_state, reward, done, _ = env.step(action)

        # store experience
        trajectory.append((state, action, reward, next_state, done))

        # move to next state
        state = next_state

    return np.array(trajectory, dtype=object)

In [45]:

def mc_control(env,
               gamma=1.0,
               init_alpha=0.5,
               min_alpha=0.01,
               alpha_decay_ratio=0.5,
               init_epsilon=1.0,
               min_epsilon=0.1,
               epsilon_decay_ratio=0.9,
               n_episodes=3000,
               max_steps=200,
               first_visit=True):

    nS, nA = env.observation_space.n, env.action_space.n

    discounts = np.logspace(
        0, max_steps,
        num=max_steps, base=gamma,
        endpoint=False)

    alphas = decay_schedule(
        init_alpha, min_alpha,
        alpha_decay_ratio,
        n_episodes)

    epsilons = decay_schedule(
        init_epsilon, min_epsilon,
        epsilon_decay_ratio,
        n_episodes)

    pi_track = []
    Q = np.zeros((nS, nA), dtype=np.float64)
    Q_track = np.zeros((n_episodes, nS, nA), dtype=np.float64)

    select_action = lambda state, Q, epsilon:np.argmax(Q[state]) if np.random.random() > epsilon else np.random.randint(len(Q[state]))

    for e in tqdm(range(n_episodes), leave=False):
        trajectory = generate_trajectory(select_action, Q, epsilons[e], env, max_steps)
        visited = np.zeros((nS, nA), dtype=bool)

        for t, (state, action, reward, _, _) in enumerate(trajectory):
            if visited[state][action] and first_visit:
                continue
            visited[state][action] = True

            n_steps = len(trajectory[t:])
            G = np.sum(discounts[:n_steps] * trajectory[t:, 2])
            Q[state][action] = Q[state][action] + alphas[e] * (G - Q[state][action])

        Q_track[e] = Q
        pi_track.append(np.argmax(Q, axis=1))

    v = np.max(Q, axis=1)
    pi = np.argmax(Q, axis=1)
    return Q, v, pi

In [54]:
from tqdm import tqdm
optimal_Q, optimal_V, optimal_pi = mc_control (env,n_episodes = 3000)
print('\nName: Shehan Shajahan\nRegister Number: 212223240154')
print_state_value_function(optimal_Q, P, n_cols=4, prec=2, title='Action-value function:')
print_state_value_function(optimal_V, P, n_cols=4, prec=2, title='State-value function:')
print_policy(optimal_pi, P)


Name: Shehan Shajahan
Register Number: 212223240154
Action-value function:
| 00 [0.13 0.1  0.15 0.12] | 01 [0.04 0.04 0.02 0.12] | 02 [0.03 0.09 0.04 0.01] | 03 [0.03 0.   0.   0.  ] |
| 04 [0.18 0.05 0.05 0.04] |           | 06 [0.05 0.05 0.14 0.  ] |           |
| 08 [0.06 0.09 0.03 0.23] | 09 [0.04 0.29 0.06 0.01] | 10 [0.13 0.21 0.28 0.08] |           |
|           | 13 [0.06 0.34 0.2  0.08] | 14 [0.24 0.79 0.66 0.46] |           |
State-value function:
| 00   0.15 | 01   0.12 | 02   0.09 | 03   0.03 |
| 04   0.18 |           | 06   0.14 |           |
| 08   0.23 | 09   0.29 | 10   0.28 |           |
|           | 13   0.34 | 14   0.79 |           |
Policy:
| 00      > | 01      ^ | 02      v | 03      < |
| 04      < |           | 06      > |           |
| 08      ^ | 09      v | 10      > |           |
|           | 13      v | 14      v |           |


In [55]:
# Find the probability of success and the mean return of you your policy
print('Name: Shehan Shajahan Register Number: 212223240154')
print('Reaches goal {:.2f}%. Obtains an average undiscounted return of {:.4f}.'.format(
    probability_success(env, optimal_pi, goal_state=goal_state)*100,
    mean_return(env, optimal_pi)))

Name: Shehan Shajahan Register Number: 212223240154
Reaches goal 17.00%. Obtains an average undiscounted return of 0.1700.
